# Patches — Bloc 4a, Sessió 6
### Estructura MIDI i `mido`

**Com obrir aquest notebook:** Fes clic a l'enllaç del Classroom. Per guardar els teus canvis: `File → Save a copy in Drive`.

Aquest notebook és per a la **demo guiada en directe**. La reproducció de fitxers MIDI i la connexió a ports MIDI reals es fa a **Thonny** (`exemples.py`). Aquí treballem la lectura, anàlisi i creació de fitxers `.mid`.

In [ ]:
!apt-get install -y fluidsynth fluid-soundfont-gm -q > /dev/null
!pip install mido pretty_midi pyfluidsynth -q
import mido
import pretty_midi
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display
import urllib.request

# Descarreguem el fitxer MIDI d'exemple
base_url = "https://raw.githubusercontent.com/jcomajuncosas/tp2/main/04_midi/sessio06_estructura_midi/recursos/midi/"
urllib.request.urlretrieve(base_url + "example_scale.mid", "example_scale.mid")
print("example_scale.mid descarregat")

## 1. Llegir un fitxer MIDI

In [ ]:
mid = mido.MidiFile('example_scale.mid')

print(f"Tracks: {len(mid.tracks)}")
print(f"Ticks per beat: {mid.ticks_per_beat}")
print(f"Durada total: {mid.length:.2f} s")

In [ ]:
# Inspeccionem els missatges de cada track
for i, track in enumerate(mid.tracks):
    print(f"\n=== Track {i}: '{track.name}' ({len(track)} missatges) ===")
    for msg in track:
        print(f"  {msg}")

## 2. Analitzar els missatges: extreure notes

🎤 *Com extraiem les notes (número, velocity, durada) d'un fitxer MIDI?*

In [ ]:
# Extraiem les notes del track de melodia (track 1)
# Necessitem aparellar cada note_on amb el seu note_off

def extrau_notes(track, ticks_per_beat, tempo=500000):
    """Retorna llista de (note, velocity, start_s, end_s)"""
    notes = []
    notes_actives = {}  # note -> (velocity, start_ticks)
    ticks_acumulats = 0

    for msg in track:
        ticks_acumulats += msg.time

        if msg.type == 'note_on' and msg.velocity > 0:
            notes_actives[msg.note] = (msg.velocity, ticks_acumulats)

        elif msg.type == 'note_off' or (msg.type == 'note_on' and msg.velocity == 0):
            if msg.note in notes_actives:
                vel, start_ticks = notes_actives.pop(msg.note)
                start_s = mido.tick2second(start_ticks, ticks_per_beat, tempo)
                end_s   = mido.tick2second(ticks_acumulats, ticks_per_beat, tempo)
                notes.append((msg.note, vel, start_s, end_s))

    return notes

# Obtenim el tempo del track de metadades
tempo = 500000  # default 120bpm
for msg in mid.tracks[0]:
    if msg.type == 'set_tempo':
        tempo = msg.tempo

notes = extrau_notes(mid.tracks[1], mid.ticks_per_beat, tempo)

print(f"{'Nota':>6} {'Nom':>6} {'Velocity':>10} {'Inici(s)':>10} {'Fi(s)':>8} {'Durada(s)':>10}")
print("-" * 55)
noms = ['Do','Do#','Re','Re#','Mi','Fa','Fa#','Sol','Sol#','La','La#','Si']
for note, vel, start, end in notes:
    nom = f"{noms[note%12]}{note//12-1}"
    print(f"{note:>6} {nom:>6} {vel:>10} {start:>10.3f} {end:>8.3f} {end-start:>10.3f}")

## 3. Visualitzar el piano roll

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

noms = ['Do','Do#','Re','Re#','Mi','Fa','Fa#','Sol','Sol#','La','La#','Si']
colors = ['steelblue', 'coral']

for t_idx, track in enumerate([mid.tracks[1], mid.tracks[2]]):
    track_notes = extrau_notes(track, mid.ticks_per_beat, tempo)
    for note, vel, start, end in track_notes:
        ax.barh(note, end - start, left=start,
                height=0.7, color=colors[t_idx], alpha=0.8)

# Etiquetes de notes
yticks = range(48, 75)
ax.set_yticks(list(yticks))
ax.set_yticklabels([f"{noms[n%12]}{n//12-1}" for n in yticks], fontsize=7)
ax.set_xlabel("Temps (s)")
ax.set_title("Piano roll — example_scale.mid")
ax.legend(['Melodia', 'Baix'], loc='upper right')
plt.tight_layout()
plt.show()

## 4. Crear un fitxer MIDI des de zero

🎤 *La mateixa lògica que `for note in pattern: piano(note, vel, dur)` de TP1.*

In [ ]:
mid_nou = mido.MidiFile(ticks_per_beat=480)
tempo = mido.bpm2tempo(120)
quarter = 480   # 1 negra
eighth  = 240   # 1 corxera

# Track de metadades
meta = mido.MidiTrack()
meta.append(mido.MetaMessage('set_tempo', tempo=tempo, time=0))
mid_nou.tracks.append(meta)

# Track de notes: acord de Do major arpegiado
track = mido.MidiTrack()
pattern = [60, 64, 67, 72, 67, 64, 60, 60]

for note in pattern:
    track.append(mido.Message('note_on',  channel=0, note=note, velocity=80, time=0))
    track.append(mido.Message('note_off', channel=0, note=note, velocity=0,  time=eighth))

track.append(mido.MetaMessage('end_of_track', time=0))
mid_nou.tracks.append(track)
mid_nou.save('nou.mid')
print(f"Creat nou.mid amb {len(pattern)} notes")

In [ ]:
# Escoltar el resultat convertint a àudio amb pretty_midi
pm = pretty_midi.PrettyMIDI('nou.mid')
audio = pm.fluidsynth(fs=44100)
Audio(audio, rate=44100)

## 5. Delta time: l'error típic

🎤 *Qué passa si posem `time=quarter` al `note_on` en lloc del `note_off`?*

In [ ]:
# Versió INCORRECTA: silenci de 1 negra ABANS de cada nota
mid_bug = mido.MidiFile(ticks_per_beat=480)
meta_b = mido.MidiTrack()
meta_b.append(mido.MetaMessage('set_tempo', tempo=mido.bpm2tempo(120), time=0))
mid_bug.tracks.append(meta_b)

track_bug = mido.MidiTrack()
for note in [60, 64, 67]:
    track_bug.append(mido.Message('note_on',  note=note, velocity=80, time=480))  # BUG!
    track_bug.append(mido.Message('note_off', note=note, velocity=0,  time=480))
track_bug.append(mido.MetaMessage('end_of_track', time=0))
mid_bug.tracks.append(track_bug)
mid_bug.save('bug.mid')

pm_bug = pretty_midi.PrettyMIDI('bug.mid')
audio_bug = pm_bug.fluidsynth(fs=44100)
print("Versió amb bug (silencis inesperats entre notes):")
display(Audio(audio_bug, rate=44100))

# Versió CORRECTA
mid_ok = mido.MidiFile(ticks_per_beat=480)
meta_ok = mido.MidiTrack()
meta_ok.append(mido.MetaMessage('set_tempo', tempo=mido.bpm2tempo(120), time=0))
mid_ok.tracks.append(meta_ok)

track_ok = mido.MidiTrack()
for note in [60, 64, 67]:
    track_ok.append(mido.Message('note_on',  note=note, velocity=80, time=0))    # OK
    track_ok.append(mido.Message('note_off', note=note, velocity=0,  time=480))
track_ok.append(mido.MetaMessage('end_of_track', time=0))
mid_ok.tracks.append(track_ok)
mid_ok.save('ok.mid')

pm_ok = pretty_midi.PrettyMIDI('ok.mid')
audio_ok = pm_ok.fluidsynth(fs=44100)
print("Versió correcta:")
display(Audio(audio_ok, rate=44100))

## 6. Iterar amb temps en segons (mido)

Quan iterem el fitxer sencer (no track a track), `mido` converteix els ticks a segons automàticament.

In [ ]:
mid = mido.MidiFile('example_scale.mid')

print("Missatges amb temps en SEGONS (iterant el fitxer sencer):")
for msg in mid:
    if not msg.is_meta:
        print(f"  t={msg.time:.4f}s  {msg}")